# Practice 099 — Causal DAGs & Identification Strategies

**Theoretical context**: see `CLAUDE.md` in this folder before starting.

**Phases**: this notebook mirrors the phases in `CLAUDE.md` § Instructions.
Each phase's exercise calls into a `src/_0N_<phase_name>.py` companion module —
read that module's `TODO(human)` block before implementing it there, then
re-run the corresponding cell below.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import xy.pyplot as plt

from src.datasets import GRAPHS, LAYOUTS, load_dataset
from src.plotting import bias_magnitude_plot, draw_dag, estimate_vs_truth_plot

## Phase 1 — D-separation

D-separation is the graphical test for reading conditional independencies off a
DAG. We start by looking at the four scenario graphs this practice uses
throughout, then implement the checker itself.

In [ ]:
for name in ("confounder", "mediator", "collider", "mbias"):
    fig = draw_dag(GRAPHS[name], LAYOUTS[name], title=name)
    fig

### Exercise — `src/_01_dsep.py :: is_d_separated`

Open `src/_01_dsep.py`, read the `TODO(human)` block above the function,
implement it there (not in this cell), then re-run the cell below.

In [ ]:
from src._01_dsep import is_d_separated

print("confounder, no adjustment  ->", is_d_separated(GRAPHS["confounder"], "D", "Y", set()), "(expect False)")
print("confounder, adjust for Z   ->", is_d_separated(GRAPHS["confounder"], "D", "Y", {"Z"}), "(expect True)")
print("collider, no adjustment    ->", is_d_separated(GRAPHS["collider"], "D", "Y", set()), "(expect True)")
print("collider, adjust for C     ->", is_d_separated(GRAPHS["collider"], "D", "Y", {"C"}), "(expect False)")

## Phase 2 — The backdoor criterion

The backdoor criterion turns "adjust for confounders, not mediators or
colliders" into one graphical test built on top of Phase 1's `is_d_separated`.

### Exercise — `src/_02_backdoor.py :: is_valid_backdoor_set`

Open `src/_02_backdoor.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._02_backdoor import is_valid_backdoor_set

print("confounder, {}   ->", is_valid_backdoor_set(GRAPHS["confounder"], "D", "Y", set()), "(expect False)")
print("confounder, {Z}  ->", is_valid_backdoor_set(GRAPHS["confounder"], "D", "Y", {"Z"}), "(expect True)")
print("mediator,   {M}  ->", is_valid_backdoor_set(GRAPHS["mediator"], "D", "Y", {"M"}), "(expect False -- M is a descendant of D)")
print("collider,   {C}  ->", is_valid_backdoor_set(GRAPHS["collider"], "D", "Y", {"C"}), "(expect False -- C is a collider)")

## Phase 3 — Three structures, one estimator

Confounder, mediator, and collider share the same three-node shape but demand
opposite adjustment decisions. We generate all three datasets (each with a
known true effect) and compare the correct vs. the wrong adjustment set on
each, using a single regression-adjustment estimator.

In [ ]:
confounder_data = load_dataset("confounder", n=1000, seed=0)
mediator_data = load_dataset("mediator", n=1000, seed=0)
collider_data = load_dataset("collider", n=1000, seed=0)
confounder_data.df.head()

### Exercise — `src/_03_structures.py :: estimate_effect_by_adjustment`

Open `src/_03_structures.py`, read the `TODO(human)` block above the function,
implement it there, then re-run the cell below.

In [ ]:
from src._03_structures import estimate_effect_by_adjustment

scenarios = [
    ("confounder", confounder_data, ["Z"], []),
    ("mediator", mediator_data, [], ["M"]),
    ("collider", collider_data, [], ["C"]),
]
adjustment_labels, adjustment_estimates, adjustment_truth = [], [], []
for name, data, correct_set, wrong_set in scenarios:
    est_correct = estimate_effect_by_adjustment(data.df, data.treatment, data.outcome, correct_set)
    est_wrong = estimate_effect_by_adjustment(data.df, data.treatment, data.outcome, wrong_set)
    print(f"[{name}] true={data.true_effect:.3f}  adjust_for={correct_set!s:6s}->{est_correct:.3f}  adjust_for={wrong_set!s:6s}->{est_wrong:.3f}")
    adjustment_labels += [f"{name}: {correct_set or 'none'}", f"{name}: {wrong_set or 'none'}"]
    adjustment_estimates += [est_correct, est_wrong]
    adjustment_truth.append(data.true_effect)

## Phase 4 — M-bias and selection bias

M-bias hides a collider (M) behind two unmeasured causes, so it *looks* like a
confounder worth controlling for. Selection bias produces the same distortion
by filtering the sample instead of adjusting a regression. Both cases: the
unadjusted estimate is already close to the truth, and the "careful" analyst who
adds the extra control makes it worse.

In [ ]:
mbias_data = load_dataset("mbias", n=1000, seed=0)
print(f"true ATE = {mbias_data.true_effect:.3f}")
mbias_data.df.head()

### Exercise — `src/_04_mbias_selection.py :: collider_stratification_bias`

Open `src/_04_mbias_selection.py`, read the `TODO(human)` block above the
function, implement it there, then re-run the cell below.

In [ ]:
from src._04_mbias_selection import collider_stratification_bias, selection_bias_demo

mbias_result = collider_stratification_bias(mbias_data.df, "D", "Y", "M")
collider_result = collider_stratification_bias(collider_data.df, "D", "Y", "C")
selection_result = selection_bias_demo(collider_data, selection_var="C")

print(f"mbias:     unadjusted={mbias_result.unadjusted:.3f}  adjusted-for-M={mbias_result.adjusted:.3f}  bias={mbias_result.bias:+.3f}")
print(f"collider:  unadjusted={collider_result.unadjusted:.3f}  adjusted-for-C={collider_result.adjusted:.3f}  bias={collider_result.bias:+.3f}")
print(f"selection: full-sample={selection_result.unadjusted:.3f}  selected-subsample={selection_result.adjusted:.3f}  bias={selection_result.bias:+.3f}")

bias_labels = ["mbias: adjust for M", "collider: adjust for C", "collider: select on C"]
bias_values = [mbias_result.bias, collider_result.bias, selection_result.bias]

## Phase 5 — Cross-check with DoWhy (no exercise)

DoWhy searches the same graphs for a valid adjustment set using the backdoor
(and, where applicable, frontdoor) criterion — its identified estimand should
agree with what Phases 1-2's from-scratch checkers already told us.

In [ ]:
from src._05_dowhy_identification import identify_effect_dowhy

for name in ("confounder", "mediator", "collider"):
    print(f"=== {name} ===")
    print(identify_effect_dowhy(name))
    print()

## Phase 6 — Structure discovery with causal-learn (no exercise)

Given only data (no graph), the PC algorithm recovers a skeleton by testing
conditional independencies. Watch how much ambiguity remains compared to
knowing the true DAG up front.

In [ ]:
from src._06_causal_learn_discovery import discover_skeleton

for name in ("confounder", "mediator", "collider"):
    print(f"=== {name} ===")
    print(discover_skeleton(name))
    print()

## Phase 7 — End-to-end run (no exercise)

The two headline figures: effect estimates vs. the true effect as the
adjustment set varies, and the bias magnitude introduced by conditioning on
(or selecting on) a collider.

In [ ]:
true_line = confounder_data.true_effect  # confounder/mbias share TRUE_ATE; mediator/collider annotated separately below
fig = estimate_vs_truth_plot(adjustment_labels, adjustment_estimates, true_value=true_line)
fig

In [ ]:
fig = bias_magnitude_plot(bias_labels, bias_values)
fig

## Verification

Sanity-checks that must pass once every TODO is implemented.

In [ ]:
assert is_d_separated(GRAPHS["confounder"], "D", "Y", {"Z"})
assert not is_d_separated(GRAPHS["collider"], "D", "Y", {"C"})
assert is_valid_backdoor_set(GRAPHS["confounder"], "D", "Y", {"Z"})
assert not is_valid_backdoor_set(GRAPHS["collider"], "D", "Y", {"C"})
assert abs(estimate_effect_by_adjustment(confounder_data.df, "D", "Y", ["Z"]) - confounder_data.true_effect) < 0.3
assert mbias_result.bias != 0 and collider_result.bias != 0
print("OK")